In [1]:
import os
import xarray as xr

In [8]:
path_extra = "../.data/downscaling_splits_extra/"
path_base  = "../.data/downscaling_splits/"

ds_train_extra = xr.open_dataset(os.path.join(path_extra, "train_extra.nc"))
ds_val_extra   = xr.open_dataset(os.path.join(path_extra, "val_extra.nc"))
ds_test_extra  = xr.open_dataset(os.path.join(path_extra, "test_extra.nc"))

ds_train_base = xr.open_dataset(os.path.join(path_base, "train.nc"))
ds_val_base   = xr.open_dataset(os.path.join(path_base, "val.nc"))
ds_test_base  = xr.open_dataset(os.path.join(path_base, "test.nc"))

ds_train_extra

<xarray.Dataset> Size: 28MB
Dimensions:      (time: 1748, lat_coarse: 39, lon_coarse: 51, lat: 240, lon: 311)
Coordinates:
  * time         (time) datetime64[ns] 14kB 2000-06-01 2000-06-02 ... 2018-08-31
  * lat_coarse   (lat_coarse) float64 312B 46.75 46.5 46.25 ... 37.75 37.5 37.25
  * lon_coarse   (lon_coarse) float64 408B -79.75 -79.5 -79.25 ... -67.5 -67.25
  * lat          (lat) float64 2kB 46.98 46.94 46.9 46.86 ... 37.11 37.07 37.03
  * lon          (lon) float64 2kB -79.97 -79.93 -79.89 ... -67.14 -67.1 -67.06
Data variables:
    ssrd_lowres  (time, lat_coarse, lon_coarse) float32 14MB ...
    d2m_lowres   (time, lat_coarse, lon_coarse) float32 14MB ...
    valid_mask   (lat, lon) int8 75kB ...
Attributes:
    description:   train split for extra ERA5 features
    train_period:  2000-2018
    val_period:    2019-2021
    test_period:   2022-2025
    note:          Extra low-resolution ERA5 predictors on native coarse grid...

In [9]:
def add_interp_extra_features(ds_extra, ds_base):
    ds = ds_extra.copy()

    ssrd_interp = ds["ssrd_lowres"].interp(
        lat_coarse=ds_base["lat"],
        lon_coarse=ds_base["lon"],
        method="linear"
    ).rename("ssrd_lowres_interp")

    d2m_interp = ds["d2m_lowres"].interp(
        lat_coarse=ds_base["lat"],
        lon_coarse=ds_base["lon"],
        method="linear"
    ).rename("d2m_lowres_interp")

    ds["ssrd_lowres_interp"] = ssrd_interp.astype("float32")
    ds["d2m_lowres_interp"]  = d2m_interp.astype("float32")

    return ds

In [10]:
ds_train_extra = add_interp_extra_features(ds_train_extra, ds_train_base)
ds_val_extra   = add_interp_extra_features(ds_val_extra, ds_val_base)
ds_test_extra  = add_interp_extra_features(ds_test_extra, ds_test_base)

In [11]:
print(ds_train_extra["ssrd_lowres_interp"].shape)
print(ds_train_extra["d2m_lowres_interp"].shape)
ds_train_extra

(1748, 240, 311)
(1748, 240, 311)


<xarray.Dataset> Size: 1GB
Dimensions:             (time: 1748, lat_coarse: 39, lon_coarse: 51, lat: 240,
                         lon: 311)
Coordinates:
  * time                (time) datetime64[ns] 14kB 2000-06-01 ... 2018-08-31
  * lat_coarse          (lat_coarse) float64 312B 46.75 46.5 ... 37.5 37.25
  * lon_coarse          (lon_coarse) float64 408B -79.75 -79.5 ... -67.5 -67.25
  * lat                 (lat) float64 2kB 46.98 46.94 46.9 ... 37.11 37.07 37.03
  * lon                 (lon) float64 2kB -79.97 -79.93 -79.89 ... -67.1 -67.06
Data variables:
    ssrd_lowres         (time, lat_coarse, lon_coarse) float32 14MB ...
    d2m_lowres          (time, lat_coarse, lon_coarse) float32 14MB ...
    valid_mask          (lat, lon) int8 75kB ...
    ssrd_lowres_interp  (time, lat, lon) float32 522MB nan nan nan ... nan nan
    d2m_lowres_interp   (time, lat, lon) float32 522MB nan nan nan ... nan nan
Attributes:
    description:   train split for extra ERA5 features
    train_period:  2000-2018
    val_period:    2019-2021
    test_period:   2022-2025
    note:          Extra low-resolution ERA5 predictors on native coarse grid...

In [12]:
mask = ds_train_base["valid_mask"] == 1

ssrd_train_valid = ds_train_extra["ssrd_lowres_interp"].where(mask)
d2m_train_valid  = ds_train_extra["d2m_lowres_interp"].where(mask)

ssrd_mean = float(ssrd_train_valid.mean().values)
ssrd_std  = float(ssrd_train_valid.std().values)

d2m_mean = float(d2m_train_valid.mean().values)
d2m_std  = float(d2m_train_valid.std().values)

print("ssrd_mean, ssrd_std =", ssrd_mean, ssrd_std)
print("d2m_mean, d2m_std   =", d2m_mean, d2m_std)

ssrd_mean, ssrd_std = 615483.0 315204.21875
d2m_mean, d2m_std   = 154.6400146484375 99.48346710205078


In [13]:
def normalize_extra_split(ds, ssrd_mean, ssrd_std, d2m_mean, d2m_std):
    ds = ds.copy()

    ds["ssrd_norm"] = ((ds["ssrd_lowres_interp"] - ssrd_mean) / ssrd_std).astype("float32")
    ds["d2m_norm"]  = ((ds["d2m_lowres_interp"] - d2m_mean) / d2m_std).astype("float32")

    return ds


ds_train_extra_norm = normalize_extra_split(ds_train_extra, ssrd_mean, ssrd_std, d2m_mean, d2m_std)
ds_val_extra_norm   = normalize_extra_split(ds_val_extra, ssrd_mean, ssrd_std, d2m_mean, d2m_std)
ds_test_extra_norm  = normalize_extra_split(ds_test_extra, ssrd_mean, ssrd_std, d2m_mean, d2m_std)

In [14]:
def fill_invalid_with_zero_extra(ds, valid_mask):
    mask = valid_mask == 1
    ds = ds.copy()

    ds["ssrd_norm"] = ds["ssrd_norm"].where(mask, 0.0).fillna(0.0)
    ds["d2m_norm"]  = ds["d2m_norm"].where(mask, 0.0).fillna(0.0)

    return ds


ds_train_extra_norm = fill_invalid_with_zero_extra(ds_train_extra_norm, ds_train_base["valid_mask"])
ds_val_extra_norm   = fill_invalid_with_zero_extra(ds_val_extra_norm, ds_val_base["valid_mask"])
ds_test_extra_norm  = fill_invalid_with_zero_extra(ds_test_extra_norm, ds_test_base["valid_mask"])

In [15]:
for ds_name, ds in [
    ("train", ds_train_extra_norm),
    ("val", ds_val_extra_norm),
    ("test", ds_test_extra_norm),
]:
    print(ds_name)
    print("  ssrd_norm has NaN:", bool(ds["ssrd_norm"].isnull().any()))
    print("  d2m_norm has NaN :", bool(ds["d2m_norm"].isnull().any()))

train
  ssrd_norm has NaN: False
  d2m_norm has NaN : False
val
  ssrd_norm has NaN: False
  d2m_norm has NaN : False
test
  ssrd_norm has NaN: False
  d2m_norm has NaN : False


In [16]:
out_dir = "../.data/downscaling_splits_extra/"
os.makedirs(out_dir, exist_ok=True)

encoding = {
    "ssrd_lowres": {"zlib": True, "complevel": 4, "dtype": "float32"},
    "d2m_lowres": {"zlib": True, "complevel": 4, "dtype": "float32"},
    "ssrd_lowres_interp": {"zlib": True, "complevel": 4, "dtype": "float32"},
    "d2m_lowres_interp": {"zlib": True, "complevel": 4, "dtype": "float32"},
    "ssrd_norm": {"zlib": True, "complevel": 4, "dtype": "float32"},
    "d2m_norm": {"zlib": True, "complevel": 4, "dtype": "float32"},
    "valid_mask": {"zlib": True, "complevel": 4, "dtype": "int8"},
}

ds_train_extra_norm.to_netcdf(
    os.path.join(out_dir, "train_extra_norm.nc"),
    engine="netcdf4",
    encoding=encoding
)

ds_val_extra_norm.to_netcdf(
    os.path.join(out_dir, "val_extra_norm.nc"),
    engine="netcdf4",
    encoding=encoding
)

ds_test_extra_norm.to_netcdf(
    os.path.join(out_dir, "test_extra_norm.nc"),
    engine="netcdf4",
    encoding=encoding
)

In [17]:
norm_stats_extra = xr.Dataset(
    {
        "ssrd_mean": xr.DataArray(ssrd_mean),
        "ssrd_std": xr.DataArray(ssrd_std),
        "d2m_mean": xr.DataArray(d2m_mean),
        "d2m_std": xr.DataArray(d2m_std),
    }
)

norm_stats_extra.to_netcdf("../.data/downscaling_splits_extra/norm_stats_extra.nc")
norm_stats_extra

<xarray.Dataset> Size: 32B
Dimensions:    ()
Data variables:
    ssrd_mean  float64 8B 6.155e+05
    ssrd_std   float64 8B 3.152e+05
    d2m_mean   float64 8B 154.6
    d2m_std    float64 8B 99.48

In [18]:
ds_check = xr.open_dataset("../.data/downscaling_splits_extra/train_extra_norm.nc")
ds_check

<xarray.Dataset> Size: 2GB
Dimensions:             (time: 1748, lat_coarse: 39, lon_coarse: 51, lat: 240,
                         lon: 311)
Coordinates:
  * time                (time) datetime64[ns] 14kB 2000-06-01 ... 2018-08-31
  * lat_coarse          (lat_coarse) float64 312B 46.75 46.5 ... 37.5 37.25
  * lon_coarse          (lon_coarse) float64 408B -79.75 -79.5 ... -67.5 -67.25
  * lat                 (lat) float64 2kB 46.98 46.94 46.9 ... 37.11 37.07 37.03
  * lon                 (lon) float64 2kB -79.97 -79.93 -79.89 ... -67.1 -67.06
Data variables:
    ssrd_lowres         (time, lat_coarse, lon_coarse) float32 14MB ...
    d2m_lowres          (time, lat_coarse, lon_coarse) float32 14MB ...
    valid_mask          (lat, lon) int8 75kB ...
    ssrd_lowres_interp  (time, lat, lon) float32 522MB ...
    d2m_lowres_interp   (time, lat, lon) float32 522MB ...
    ssrd_norm           (time, lat, lon) float32 522MB ...
    d2m_norm            (time, lat, lon) float32 522MB ...
Attributes:
    description:   train split for extra ERA5 features
    train_period:  2000-2018
    val_period:    2019-2021
    test_period:   2022-2025
    note:          Extra low-resolution ERA5 predictors on native coarse grid...